# E6 — YOLO26s + SCConv + DySample trên DataSet2

Notebook này là thí nghiệm ablation E6 độc lập. E6 được khởi tạo từ cùng `yolo26s.pt`, giữ nguyên DataSet2, class order và protocol E0–E5. Chỉ thêm hai cải tiến: DySample tại hai upsample neck và SCConv bên trong đúng hai `C3k2` neck tạo P4/16, P3/8.

Vị trí module được dò từ runtime graph của Ultralytics 8.4.155; notebook không dùng layer index hard-code của YOLO khác. Chỉ dùng `split='val'` cho model selection, không dùng test set.


In [ ]:
%pip install -q ultralytics==8.4.155
%pip install -q gdown
!apt-get update -qq
!apt-get install -y unrar


In [ ]:
from google.colab import drive
import os
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import ultralytics
import yaml
from PIL import Image
from ultralytics import YOLO

drive.mount('/content/drive', force_remount=True)
os.makedirs('/content/DetectedHelmets/DataSet2', exist_ok=True)
os.chdir('/content/DetectedHelmets')

data_root = Path('/content/DetectedHelmets/DataSet2')
drive_dataset_path = Path('/content/drive/MyDrive/DataSet2')
if drive_dataset_path.exists() and not data_root.exists():
    data_root.symlink_to(drive_dataset_path, target_is_directory=True)
elif not drive_dataset_path.exists():
    import gdown
    archive_path = Path('/content/DetectedHelmets/dataset.rar')
    file_id = '1LUUQLl5F3aczuny4As_PcTtqLIWTmZSJ'
    gdown.download(f'https://drive.google.com/uc?id={file_id}', str(archive_path), quiet=False, fuzzy=True)
    os.system(f'unrar x -o+ {archive_path} {data_root}/')

DATA_YAML_PATH = Path('/content/DetectedHelmets/DataSet2/DataSet2/data.yaml')
MODEL_NAME = 'yolo26s.pt'
ULTRALYTICS_VERSION = '8.4.155'
OUTPUT_DIR = Path('/content/drive/MyDrive/DetectedHelmets/runs/train')
EXPERIMENT_NAME = 'helmet_yolo26s_scconv_dysample_dataset2'
EXPERIMENT_DIR = OUTPUT_DIR / EXPERIMENT_NAME
BEST_MODEL_PATH = EXPERIMENT_DIR / 'weights' / 'best.pt'
BASELINE_EXPERIMENT_NAME = 'helmet_yolo26s_dataset2'
BASELINE_BEST_MODEL_PATH = OUTPUT_DIR / BASELINE_EXPERIMENT_NAME / 'weights' / 'best.pt'
E4_BEST_MODEL_PATH = OUTPUT_DIR / 'helmet_yolo26s_dysample_dataset2' / 'weights' / 'best.pt'
E5_BEST_MODEL_PATH = OUTPUT_DIR / 'helmet_yolo26s_scconv_dataset2' / 'weights' / 'best.pt'
VALIDATION_SPLIT = 'val'
TRAIN_CONFIG = {
    'epochs': 50,
    'imgsz': 640,
    'batch': 16,
    'device': 0,
    'workers': 4,
    'optimizer': 'auto',
    'seed': 0,
    'deterministic': True,
    'patience': 15,
    'pretrained': True,
}

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Ultralytics:', ultralytics.__version__)
assert torch.cuda.is_available(), 'Protocol yêu cầu device=0 và GPU phải được bật.'
assert ultralytics.__version__ == ULTRALYTICS_VERSION == '8.4.155'
assert DATA_YAML_PATH.exists(), DATA_YAML_PATH

with DATA_YAML_PATH.open('r', encoding='utf-8') as file:
    data_config = yaml.safe_load(file)
raw_names = data_config.get('names', [])
if isinstance(raw_names, dict):
    raw_names = [raw_names[key] for key in sorted(raw_names, key=lambda item: int(item))]
CLASS_NAMES = list(raw_names)
EXPECTED_CLASS_NAMES = ['motorcycle', 'rider_with_helmet', 'rider_without_helmet']
assert data_config.get('nc') == 3
assert CLASS_NAMES == EXPECTED_CLASS_NAMES, (CLASS_NAMES, EXPECTED_CLASS_NAMES)
print('Dataset:', DATA_YAML_PATH)
print('Classes:', CLASS_NAMES)
print('Training protocol:', TRAIN_CONFIG)


## 1. Dò runtime graph và chọn hai upsample neck

`P5 -> P4` và `P4 -> P3` được xác định bằng forward hook: upsample ×2 trước Detect, với output stride lần lượt 16 và 8. Nếu runtime không có đúng hai node này, notebook dừng thay vì thay nhầm layer.


In [ ]:
base_model = YOLO(MODEL_NAME)
assert getattr(base_model, 'task', None) == 'detect', base_model.task
print('--- YOLO26s runtime architecture ---')
print(base_model.model)
base_model.info(verbose=True, imgsz=640)

def is_scale_two_upsample(module):
    if not isinstance(module, nn.Upsample):
        return False
    scale_factor = module.scale_factor
    if isinstance(scale_factor, (tuple, list)):
        return len(scale_factor) == 2 and all(float(value) == 2.0 for value in scale_factor)
    return float(scale_factor) == 2.0

def probe_runtime_upsamples(yolo_model, imgsz=640):
    records, handles = [], []
    def make_hook(index, module):
        def hook(_module, inputs, output):
            input_tensor = inputs[0] if inputs and torch.is_tensor(inputs[0]) else None
            if input_tensor is None or not torch.is_tensor(output):
                return
            input_shape, output_shape = tuple(input_tensor.shape), tuple(output.shape)
            records.append({
                'index': index,
                'from': getattr(module, 'f', None),
                'input_shape': input_shape,
                'output_shape': output_shape,
                'input_channels': input_shape[1],
                'input_stride': imgsz // input_shape[-1],
                'output_stride': imgsz // output_shape[-1],
            })
        return hook
    for index, module in enumerate(yolo_model.model.model):
        if is_scale_two_upsample(module):
            handles.append(module.register_forward_hook(make_hook(index, module)))
    device = next(yolo_model.model.parameters()).device
    was_training = yolo_model.model.training
    yolo_model.model.eval()
    with torch.no_grad():
        yolo_model.model(torch.zeros(1, 3, imgsz, imgsz, device=device))
    if was_training:
        yolo_model.model.train()
    for handle in handles:
        handle.remove()
    return sorted(records, key=lambda item: item['index'])

detect_indices = [
    index for index, module in enumerate(base_model.model.model)
    if module.__class__.__name__ in {'Detect', 'WorldDetect', 'YOLOEDetect'}
]
assert detect_indices, 'Không tìm thấy Detect head trong runtime graph.'
detect_index = min(detect_indices)
DETECT_HEAD_COUNT = len(detect_indices)
runtime_upsample_records = probe_runtime_upsamples(base_model)
print('--- Runtime nn.Upsample(scale_factor=2) ---')
for record in runtime_upsample_records:
    print(record)

neck_upsample_records = [
    record for record in runtime_upsample_records
    if record['index'] < detect_index
    and record['input_stride'] == record['output_stride'] * 2
    and record['output_stride'] in (8, 16)
]
by_output_stride = {}
for record in neck_upsample_records:
    if record['output_stride'] in by_output_stride:
        raise RuntimeError(f'Nhiều upsample neck cùng output stride: {record}')
    by_output_stride[record['output_stride']] = record
assert len(neck_upsample_records) == 2 and set(by_output_stride) == {8, 16}, neck_upsample_records

DYSAMPLE_EXPECTED_COUNT = 2
upsample_specs = [
    {
        'level': 'P5 -> P4' if output_stride == 16 else 'P4 -> P3',
        **by_output_stride[output_stride],
    }
    for output_stride in (16, 8)
]
assert len(upsample_specs) == DYSAMPLE_EXPECTED_COUNT
print('Detect index:', detect_index)
print('Selected DySample targets:', upsample_specs)


## 2. DySample — implementation đã kiểm tra ở E4

Đây là implementation theo *Learning to Upsample by Learning to Sample* (ICCV 2023), giữ `scale=2`, `style='lp'`, `groups=4` khi hợp lệ và `dyscope=True`.


In [ ]:
def normal_init(module, mean=0, std=1, bias=0):
    if hasattr(module, 'weight') and module.weight is not None:
        nn.init.normal_(module.weight, mean, std)
    if hasattr(module, 'bias') and module.bias is not None:
        nn.init.constant_(module.bias, bias)

def constant_init(module, val, bias=0):
    if hasattr(module, 'weight') and module.weight is not None:
        nn.init.constant_(module.weight, val)
    if hasattr(module, 'bias') and module.bias is not None:
        nn.init.constant_(module.bias, bias)

class DySample(nn.Module):
    def __init__(self, in_channels, scale=2, style='lp', groups=4, dyscope=False):
        super().__init__()
        self.scale, self.style, self.groups = scale, style, groups
        assert style in ['lp', 'pl']
        if style == 'pl':
            assert in_channels >= scale ** 2 and in_channels % scale ** 2 == 0
        assert in_channels >= groups and in_channels % groups == 0
        if style == 'pl':
            in_channels = in_channels // scale ** 2
            out_channels = 2 * groups
        else:
            out_channels = 2 * groups * scale ** 2
        self.offset = nn.Conv2d(in_channels, out_channels, 1)
        normal_init(self.offset, std=0.001)
        if dyscope:
            self.scope = nn.Conv2d(in_channels, out_channels, 1, bias=False)
            constant_init(self.scope, val=0.)
        self.register_buffer('init_pos', self._init_pos())

    def _init_pos(self):
        h = torch.arange((-self.scale + 1) / 2, (self.scale - 1) / 2 + 1) / self.scale
        return torch.stack(torch.meshgrid([h, h])).transpose(1, 2).repeat(1, self.groups, 1).reshape(1, -1, 1, 1)

    def sample(self, x, offset):
        batch, _, height, width = offset.shape
        offset = offset.reshape(batch, 2, -1, height, width)
        coords_h = torch.arange(height)
        coords_w = torch.arange(width)
        coords = torch.stack(torch.meshgrid([coords_w, coords_h])).transpose(1, 2).unsqueeze(1).unsqueeze(0).type(x.dtype).to(x.device)
        normalizer = torch.tensor([width, height], dtype=x.dtype, device=x.device).reshape(1, 2, 1, 1, 1)
        coords = 2 * (coords + offset) / normalizer - 1
        coords = torch.nn.functional.pixel_shuffle(coords.contiguous().reshape(batch, -1, height, width), self.scale)
        coords = coords.reshape(batch, 2, -1, self.scale * height, self.scale * width).permute(0, 2, 3, 4, 1).contiguous().flatten(0, 1)
        return torch.nn.functional.grid_sample(
            x.reshape(batch * self.groups, -1, height, width), coords,
            mode='bilinear', align_corners=False, padding_mode='border',
        ).reshape(batch, -1, self.scale * height, self.scale * width)

    def forward_lp(self, x):
        if hasattr(self, 'scope'):
            offset = self.offset(x) * self.scope(x).sigmoid() * 0.5 + self.init_pos
        else:
            offset = self.offset(x) * 0.25 + self.init_pos
        return self.sample(x, offset)

    def forward_pl(self, x):
        x_ = torch.nn.functional.pixel_shuffle(x, self.scale)
        if hasattr(self, 'scope'):
            offset = torch.nn.functional.pixel_unshuffle(self.offset(x_) * self.scope(x_).sigmoid(), self.scale) * 0.5 + self.init_pos
        else:
            offset = torch.nn.functional.pixel_unshuffle(self.offset(x_), self.scale) * 0.25 + self.init_pos
        return self.sample(x, offset)

    def forward(self, x):
        return self.forward_pl(x) if self.style == 'pl' else self.forward_lp(x)

def choose_groups(in_channels, preferred=4):
    divisors = [group for group in range(1, in_channels + 1) if in_channels % group == 0]
    return min(divisors, key=lambda group: (abs(group - preferred), -group))

for spec in upsample_specs:
    spec['groups'] = choose_groups(spec['input_channels'], preferred=4)
    spec['scale'] = 2
    spec['style'] = 'lp'
    spec['dyscope'] = True
    print(spec['level'], 'channels=', spec['input_channels'], 'groups=', spec['groups'])

unit_module = DySample(64, scale=2, style='lp', groups=4, dyscope=True)
unit_input = torch.randn(2, 64, 20, 20, requires_grad=True)
unit_output = unit_module(unit_input)
assert unit_output.shape == (2, 64, 40, 40)
unit_output.mean().backward()
assert any(parameter.grad is not None for parameter in unit_module.parameters())
assert unit_module.offset.out_channels == 2 * 4 * 2 ** 2
print('DySample unit shape/gradient check: OK')


## 3. SCConv và C3k2_SCConv — implementation đã kiểm tra ở E5

SCConv giữ nguyên chuỗi `SRU -> CRU` và cấu hình E5. `ScConv` thay convolution thứ hai của `Bottleneck`; không gắn SCConv sau C3k2 và không thay backbone C3k2.


In [ ]:
SCONV_CONFIG = {
    'group_num': 4,
    'gate_threshold': 0.5,
    'alpha': 0.5,
    'squeeze_ratio': 2,
    'group_size': 2,
    'group_kernel_size': 3,
}

def nearest_valid_group_num(channels, requested):
    valid = [group for group in range(1, channels + 1) if channels % group == 0]
    return min(valid, key=lambda group: (abs(group - requested), -group))

class GroupBatchnorm2d(nn.Module):
    def __init__(self, c_num, group_num=4, eps=1e-5):
        super().__init__()
        self.group_num = nearest_valid_group_num(c_num, group_num)
        self.weight = nn.Parameter(torch.ones(c_num, 1, 1))
        self.bias = nn.Parameter(torch.zeros(c_num, 1, 1))
        self.eps = eps

    def forward(self, x):
        batch, channels, height, width = x.shape
        grouped = x.reshape(batch, self.group_num, -1)
        mean = grouped.mean(2, keepdim=True)
        variance = grouped.var(2, keepdim=True, unbiased=False)
        normalized = ((grouped - mean) / torch.sqrt(variance + self.eps)).reshape(batch, channels, height, width)
        return normalized * self.weight + self.bias

class SRU(nn.Module):
    def __init__(self, oup_channels, group_num=4, gate_threshold=0.5):
        super().__init__()
        if oup_channels % 2:
            raise ValueError('SRU cross reconstruction requires even channels.')
        self.gn = GroupBatchnorm2d(oup_channels, group_num)
        self.gate_threshold = gate_threshold
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        gn_x = self.gn(x)
        gamma = self.gn.weight
        gamma_weight = (gamma / (gamma.sum() + torch.finfo(gamma.dtype).eps)).view(1, -1, 1, 1)
        weights = self.sigmoid(gn_x * gamma_weight)
        informative = torch.where(weights > self.gate_threshold, torch.ones_like(weights), weights)
        non_informative = torch.where(weights > self.gate_threshold, torch.zeros_like(weights), weights)
        return self.reconstruct(informative * x, non_informative * x)

    def reconstruct(self, x1, x2):
        half = x1.shape[1] // 2
        x11, x12 = torch.split(x1, [half, half], 1)
        x21, x22 = torch.split(x2, [half, half], 1)
        return torch.cat([x11 + x22, x12 + x21], 1)

class CRU(nn.Module):
    def __init__(self, op_channel, alpha=0.5, squeeze_ratio=2, group_size=2, group_kernel_size=3):
        super().__init__()
        self.up_channel = int(alpha * op_channel)
        self.low_channel = op_channel - self.up_channel
        self.up_squeeze = self.up_channel // squeeze_ratio
        self.low_squeeze = self.low_channel // squeeze_ratio
        if min(self.up_squeeze, self.low_squeeze) < 1 or self.up_squeeze % group_size or op_channel % group_size:
            raise ValueError('CRU channel configuration is incompatible.')
        self.squeeze1 = nn.Conv2d(self.up_channel, self.up_squeeze, 1, bias=False)
        self.squeeze2 = nn.Conv2d(self.low_channel, self.low_squeeze, 1, bias=False)
        self.GWC = nn.Conv2d(self.up_squeeze, op_channel, group_kernel_size, 1, group_kernel_size // 2, groups=group_size)
        self.PWC1 = nn.Conv2d(self.up_squeeze, op_channel, 1, bias=False)
        self.PWC2 = nn.Conv2d(self.low_squeeze, op_channel - self.low_squeeze, 1, bias=False)
        self.advavg = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        upper, lower = torch.split(x, [self.up_channel, self.low_channel], 1)
        upper, lower = self.squeeze1(upper), self.squeeze2(lower)
        upper_branch = self.GWC(upper) + self.PWC1(upper)
        lower_branch = torch.cat([self.PWC2(lower), lower], 1)
        fused = torch.cat([upper_branch, lower_branch], 1)
        fused = F.softmax(self.advavg(fused), dim=1) * fused
        first, second = torch.split(fused, fused.shape[1] // 2, 1)
        return first + second

class ScConv(nn.Module):
    def __init__(self, op_channel, **kwargs):
        super().__init__()
        self.SRU = SRU(op_channel, kwargs.get('group_num', 4), kwargs.get('gate_threshold', 0.5))
        self.CRU = CRU(op_channel, kwargs.get('alpha', 0.5), kwargs.get('squeeze_ratio', 2), kwargs.get('group_size', 2), kwargs.get('group_kernel_size', 3))

    def forward(self, x):
        return self.CRU(self.SRU(x))

scconv_input = torch.randn(2, 32, 16, 16, requires_grad=True)
scconv_unit = ScConv(32, **SCONV_CONFIG)
scconv_output = scconv_unit(scconv_input)
assert scconv_output.shape == scconv_input.shape
scconv_output.mean().backward()
assert any(parameter.grad is not None for parameter in scconv_unit.parameters())
print('SCConv unit shape/gradient check: OK')


In [ ]:
from ultralytics.nn.modules.block import Bottleneck

def wrapped_channels(wrapper):
    raw = getattr(wrapper, 'conv', wrapper)
    return int(raw.in_channels), int(raw.out_channels)

def wrapped_stride(wrapper):
    stride = getattr(getattr(wrapper, 'conv', wrapper), 'stride', 1)
    return (stride, stride) if isinstance(stride, int) else tuple(stride)

class Bottleneck_SCConv(nn.Module):
    def __init__(self, original):
        super().__init__()
        self.cv1 = original.cv1
        self.add = bool(getattr(original, 'add', False))
        input_channels, output_channels = wrapped_channels(original.cv2)
        self.scconv_applied = (
            wrapped_stride(original.cv2) == (1, 1)
            and input_channels == output_channels
            and input_channels % 2 == 0
        )
        self.cv2 = ScConv(input_channels, **SCONV_CONFIG) if self.scconv_applied else original.cv2

    def forward(self, x):
        output = self.cv2(self.cv1(x))
        return x + output if self.add else output

def replace_bottlenecks(module, stats):
    if module.__class__.__name__ == 'Bottleneck':
        replacement = Bottleneck_SCConv(module)
        stats['seen'] += 1
        stats['scconv'] += int(replacement.scconv_applied)
        stats['kept'] += int(not replacement.scconv_applied)
        return replacement
    for name, child in list(module.named_children()):
        replacement = replace_bottlenecks(child, stats)
        if replacement is not child:
            setattr(module, name, replacement)
    return module

class C3k2_SCConv(nn.Module):
    def __init__(self, original):
        super().__init__()
        self.c = original.c
        self.cv1, self.cv2 = original.cv1, original.cv2
        self.stats = {'seen': 0, 'scconv': 0, 'kept': 0}
        self.m = replace_bottlenecks(original.m, self.stats)

    def forward(self, x):
        values = list(self.cv1(x).chunk(2, 1))
        values.extend(block(values[-1]) for block in self.m)
        return self.cv2(torch.cat(values, 1))

    def forward_split(self, x):
        values = list(self.cv1(x).split((self.c, self.c), 1))
        values.extend(block(values[-1]) for block in self.m)
        return self.cv2(torch.cat(values, 1))

def copy_graph_metadata(replacement, old_module, fallback_index, fallback_from):
    replacement.i = getattr(old_module, 'i', fallback_index)
    replacement.f = getattr(old_module, 'f', fallback_from)
    replacement.type = replacement.__class__.__name__
    replacement.np = sum(parameter.numel() for parameter in replacement.parameters())
    return replacement

def count_dysample(module):
    return sum(1 for item in module.modules() if item.__class__.__name__ == 'DySample')

def count_c3k2_scconv(module):
    return sum(1 for item in module.modules() if item.__class__.__name__ == 'C3k2_SCConv')

def count_scconv(module):
    return sum(1 for item in module.modules() if item.__class__.__name__ == 'ScConv')

def sum_tensors(value):
    if torch.is_tensor(value):
        return value.float().sum()
    if isinstance(value, (list, tuple)):
        terms = [sum_tensors(item) for item in value]
        return sum(terms) if terms else torch.zeros(())
    if isinstance(value, dict):
        terms = [sum_tensors(item) for item in value.values()]
        return sum(terms) if terms else torch.zeros(())
    return torch.zeros(())


## 4. Chèn DySample trước fusion và SCConv vào đúng hai C3k2 neck

Trước tiên chèn DySample vào hai `nn.Upsample` đã dò. Sau đó dò lại `C3k2` trên runtime graph mới và chỉ chọn node là input trực tiếp của Detect ở stride 16 và 8. E6 không thay các C3k2 của backbone.


In [ ]:
def replace_runtime_upsamples(yolo_model, specs):
    for spec in specs:
        old_module = yolo_model.model.model[spec['index']]
        assert is_scale_two_upsample(old_module), (spec, old_module.__class__.__name__)
        replacement = DySample(
            spec['input_channels'], scale=spec['scale'], style=spec['style'],
            groups=spec['groups'], dyscope=spec['dyscope'],
        )
        yolo_model.model.model[spec['index']] = copy_graph_metadata(replacement, old_module, spec['index'], spec['from'])
    return yolo_model

def class_count(module, class_name):
    return sum(1 for item in module.modules() if item.__class__.__name__ == class_name)

def probe_c3k2(yolo_model, imgsz=640):
    records, handles = [], []
    def make_hook(index, module):
        def hook(_module, inputs, output):
            if not torch.is_tensor(output):
                return
            input_value = inputs[0] if inputs else None
            records.append({
                'index': index,
                'from': getattr(module, 'f', None),
                'input_shape': tuple(input_value.shape) if torch.is_tensor(input_value) else None,
                'output_shape': tuple(output.shape),
                'stride': imgsz // output.shape[-1],
                'bottleneck_count': class_count(module, 'Bottleneck'),
            })
        return hook
    for index, module in enumerate(yolo_model.model.model):
        if module.__class__.__name__ == 'C3k2':
            handles.append(module.register_forward_hook(make_hook(index, module)))
    device = next(yolo_model.model.parameters()).device
    was_training = yolo_model.model.training
    yolo_model.model.eval()
    with torch.no_grad():
        yolo_model.model(torch.zeros(1, 3, imgsz, imgsz, device=device))
    if was_training:
        yolo_model.model.train()
    for handle in handles:
        handle.remove()
    return sorted(records, key=lambda item: item['index'])

def build_custom_dysample_model():
    return replace_runtime_upsamples(YOLO(MODEL_NAME), upsample_specs)

def build_custom_scconv_model():
    model = YOLO(MODEL_NAME)
    for spec in c3k2_specs:
        old = model.model.model[spec['index']]
        model.model.model[spec['index']] = copy_graph_metadata(C3k2_SCConv(old), old, spec['index'], spec['from'])
    return model

def build_custom_e6_model():
    model = replace_runtime_upsamples(YOLO(MODEL_NAME), upsample_specs)
    detect_entries = [
        (index, module) for index, module in enumerate(model.model.model)
        if module.__class__.__name__ in {'Detect', 'WorldDetect', 'YOLOEDetect'}
    ]
    detect_index, detect_head = min(detect_entries, key=lambda item: item[0])
    detect_from = getattr(detect_head, 'f', None)
    assert isinstance(detect_from, (list, tuple)), detect_from
    detect_source_indices = {int(index) for index in detect_from}
    c3k2_probe = probe_c3k2(model)
    neck_start_index = min(spec['index'] for spec in upsample_specs)
    candidates = [
        record for record in c3k2_probe
        if neck_start_index < record['index'] < detect_index
        and record['index'] in detect_source_indices
        and record['stride'] in (8, 16)
    ]
    selected_by_stride = {}
    for stride in (8, 16):
        matches = [record for record in candidates if record['stride'] == stride]
        if len(matches) != 1:
            print('C3k2 candidates:', candidates)
            print(model.model)
            raise RuntimeError(f'Không chọn được đúng một C3k2 neck ở stride {stride}.')
        selected_by_stride[stride] = matches[0]
    specs = [
        {'level': 'P4/16' if stride == 16 else 'P3/8', **selected_by_stride[stride]}
        for stride in (16, 8)
    ]
    assert len(specs) == 2
    for spec in specs:
        old = model.model.model[spec['index']]
        assert old.__class__.__name__ == 'C3k2', old.__class__.__name__
        replacement = C3k2_SCConv(old)
        print('Replacing', spec['level'], 'at runtime index', spec['index'], replacement.stats)
        model.model.model[spec['index']] = copy_graph_metadata(replacement, old, spec['index'], spec['from'])
    return model, specs

def build_custom_e6_graph():
    model, _ = build_custom_e6_model()
    return model

custom_model, c3k2_specs = build_custom_e6_model()
C3K2_SCCONV_EXPECTED_COUNT = 2
assert count_dysample(custom_model.model) == DYSAMPLE_EXPECTED_COUNT == 2
assert count_c3k2_scconv(custom_model.model) == C3K2_SCCONV_EXPECTED_COUNT == 2
assert count_scconv(custom_model.model) > 0

source_state = base_model.model.state_dict()
target_state = custom_model.model.state_dict()
compatible_state = {
    key: value for key, value in source_state.items()
    if key in target_state and tuple(value.shape) == tuple(target_state[key].shape)
}
load_report = custom_model.model.load_state_dict(compatible_state, strict=False)
source_parameters = dict(base_model.model.named_parameters())
target_parameters = dict(custom_model.model.named_parameters())
transferred_parameter_keys = {
    key for key in compatible_state if key in source_parameters and key in target_parameters
}
transferred_parameters = sum(source_parameters[key].numel() for key in transferred_parameter_keys)
baseline_parameter_count = sum(parameter.numel() for parameter in base_model.model.parameters())
custom_parameter_count = sum(parameter.numel() for parameter in custom_model.model.parameters())
new_parameter_count = sum(value.numel() for key, value in target_parameters.items() if key not in transferred_parameter_keys)

print('Baseline parameters:', f'{baseline_parameter_count:,}')
print('E6 parameters:', f'{custom_parameter_count:,}')
print('New parameters:', f'{new_parameter_count:,}')
print('Transferred pretrained parameters:', f'{transferred_parameters:,}')
print('Compatible state tensors:', len(compatible_state))
print('Missing state tensors:', len(load_report.missing_keys))
print('DySample:', count_dysample(custom_model.model))
print('C3k2_SCConv:', count_c3k2_scconv(custom_model.model))
print('ScConv:', count_scconv(custom_model.model))
assert count_dysample(custom_model.model) == 2
assert count_c3k2_scconv(custom_model.model) == 2


## 5. Preflight: shape, full forward và gradient

Các hook dưới đây kiểm tra đúng bốn điểm E6: DySample P5→P4, C3k2_SCConv P4, DySample P4→P3 và C3k2_SCConv P3.


In [ ]:
shape_records, hook_handles = [], []
for index, module in enumerate(custom_model.model.model):
    if module.__class__.__name__ in {'DySample', 'C3k2_SCConv'}:
        def shape_hook(current_module, inputs, output, index=index):
            shape_records.append({
                'index': index,
                'module': current_module.__class__.__name__,
                'input': tuple(inputs[0].shape),
                'output': tuple(output.shape),
            })
        hook_handles.append(module.register_forward_hook(shape_hook))

device = next(custom_model.model.parameters()).device
dummy = torch.randn(2, 3, 640, 640, device=device)
custom_model.model.eval()
with torch.no_grad():
    dummy_output = custom_model.model(dummy)
for handle in hook_handles:
    handle.remove()
assert shape_records, 'Không thu được hook shape cho module E6.'
print('Hook shapes:', shape_records)
assert len([record for record in shape_records if record['module'] == 'DySample']) == 2
assert len([record for record in shape_records if record['module'] == 'C3k2_SCConv']) == 2
for record in shape_records:
    if record['module'] == 'DySample':
        assert record['output'][1] == record['input'][1]
        assert record['output'][2] == record['input'][2] * 2
        assert record['output'][3] == record['input'][3] * 2
assert dummy_output is not None
print('Full YOLO dummy forward: OK')

custom_model.model.train()
custom_model.model.zero_grad(set_to_none=True)
graph_output = custom_model.model(torch.randn(2, 3, 640, 640, device=device, requires_grad=True))
graph_loss = sum_tensors(graph_output)
assert graph_loss.requires_grad, 'Output không còn nằm trong computational graph.'
graph_loss.backward()
dysample_gradients = [
    parameter.grad for item in custom_model.model.modules()
    if item.__class__.__name__ == 'DySample'
    for parameter in item.parameters() if parameter.grad is not None
]
scconv_gradients = [
    parameter.grad for item in custom_model.model.modules()
    if item.__class__.__name__ == 'ScConv'
    for parameter in item.parameters() if parameter.grad is not None
]
dysample_grad_ok = bool(dysample_gradients) and all(torch.isfinite(gradient).all() for gradient in dysample_gradients)
scconv_grad_ok = bool(scconv_gradients) and all(torch.isfinite(gradient).all() for gradient in scconv_gradients)
print('dysample_grad_ok:', dysample_grad_ok)
print('scconv_grad_ok:', scconv_grad_ok)
assert dysample_grad_ok
assert scconv_grad_ok
custom_model.model.eval()


## 6. Custom DetectionTrainer và train E6

Trainer nhận trực tiếp PyTorch graph đã thay module. Trước khi train phải chưa tồn tại `EXPERIMENT_DIR`, nhằm tránh Ultralytics tự tạo hậu tố `_2`.


In [ ]:
from ultralytics.models.yolo.detect import DetectionTrainer

custom_pytorch_model = custom_model.model
dysample_before_train = count_dysample(custom_pytorch_model)
c3k2_scconv_before_train = count_c3k2_scconv(custom_pytorch_model)
print('DySample BEFORE TRAIN:', dysample_before_train)
print('C3k2_SCConv BEFORE TRAIN:', c3k2_scconv_before_train)
assert dysample_before_train == 2
assert c3k2_scconv_before_train == 2

class CustomE6Trainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        return custom_pytorch_model

    def setup_model(self):
        self.model = custom_pytorch_model
        assert count_dysample(self.model) == 2
        assert count_c3k2_scconv(self.model) == 2
        print('DySample INSIDE setup_model:', count_dysample(self.model))
        print('C3k2_SCConv INSIDE setup_model:', count_c3k2_scconv(self.model))
        return None

TRAIN_OVERRIDES = {
    'model': MODEL_NAME,
    'data': str(DATA_YAML_PATH),
    **TRAIN_CONFIG,
    'project': str(OUTPUT_DIR),
    'name': EXPERIMENT_NAME,
    'save': True,
    'save_period': 10,
    'plots': True,
    'exist_ok': False,
}
assert not EXPERIMENT_DIR.exists(), f'Output đã tồn tại; dừng để tránh tạo _2: {EXPERIMENT_DIR}'
trainer = CustomE6Trainer(overrides=TRAIN_OVERRIDES, _callbacks=custom_model.callbacks)
trainer.model = custom_pytorch_model
c3k2_scconv_inside_trainer = count_c3k2_scconv(trainer.model)
dysample_inside_trainer = count_dysample(trainer.model)
print('DySample INSIDE TRAINER:', dysample_inside_trainer)
print('C3k2_SCConv INSIDE TRAINER:', c3k2_scconv_inside_trainer)
assert dysample_inside_trainer == 2
assert c3k2_scconv_inside_trainer == 2
train_results = trainer.train()
assert EXPERIMENT_DIR.exists()


## 7. Load best checkpoint mà không làm mất module custom

Ưu tiên deserialize trực tiếp. Nếu checkpoint không deserialize được, rebuild từ `yolo26s.pt`, chèn lại đúng hai DySample + hai C3k2_SCConv, rồi load các tensor tương thích theo shape.


In [ ]:
EXPERIMENT_VERSION_FILE = EXPERIMENT_DIR / 'ultralytics_version.txt'
EXPERIMENT_VERSION_FILE.write_text(
    f'ultralytics=={ultralytics.__version__}\n'
    f'model={MODEL_NAME}\n'
    f'dysample_expected_count={DYSAMPLE_EXPECTED_COUNT}\n'
    f'c3k2_scconv_expected_count={C3K2_SCCONV_EXPECTED_COUNT}\n'
    f'upsample_specs={upsample_specs}\n'
    f'c3k2_specs={c3k2_specs}\n',
    encoding='utf-8',
)

def torch_load_checkpoint(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')

def extract_checkpoint_state(payload):
    if isinstance(payload, nn.Module):
        return payload.state_dict()
    if isinstance(payload, dict):
        for key in ('model', 'ema', 'state_dict'):
            value = payload.get(key)
            if isinstance(value, nn.Module):
                return value.state_dict()
            if isinstance(value, dict):
                tensors = {name: item for name, item in value.items() if torch.is_tensor(item)}
                if tensors:
                    return tensors
        tensors = {name: item for name, item in payload.items() if torch.is_tensor(item)}
        if tensors:
            return tensors
    raise TypeError(f'Không tìm thấy state dict trong checkpoint type={type(payload)}')

def load_arch_checkpoint(path, builder, validator, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'{label} checkpoint không tồn tại: {path}')
    try:
        direct = YOLO(str(path))
        if validator(direct.model):
            print(label, 'checkpoint load mode: direct')
            return direct, 'direct'
    except Exception as direct_error:
        print(label, 'direct load unavailable:', direct_error)
    rebuilt = builder()
    state = extract_checkpoint_state(torch_load_checkpoint(str(path)))
    if state and all(name.startswith('module.') for name in state):
        state = {name[7:]: value for name, value in state.items()}
    target = rebuilt.model.state_dict()
    compatible = {
        name: value for name, value in state.items()
        if name in target and tuple(value.shape) == tuple(target[name].shape)
    }
    if not compatible:
        raise RuntimeError(f'{label}: checkpoint không có tensor tương thích.')
    report = rebuilt.model.load_state_dict(compatible, strict=False)
    print(label, 'rebuilt compatible tensors:', len(compatible), 'missing:', len(report.missing_keys))
    assert validator(rebuilt.model)
    return rebuilt, 'rebuilt'

best_model, checkpoint_load_mode = load_arch_checkpoint(
    BEST_MODEL_PATH, build_custom_e6_graph,
    lambda module: count_dysample(module) == 2 and count_c3k2_scconv(module) == 2,
    'E6',
)
dysample_count_best = count_dysample(best_model.model)
c3k2_scconv_count_best = count_c3k2_scconv(best_model.model)
scconv_count_best = count_scconv(best_model.model)
print('DySample IN BEST MODEL:', dysample_count_best)
print('C3k2_SCConv IN BEST MODEL:', c3k2_scconv_count_best)
print('ScConv IN BEST MODEL:', scconv_count_best, 'load mode:', checkpoint_load_mode)
assert dysample_count_best == 2
assert c3k2_scconv_count_best == 2


## 8. Metrics validation, per-class report, confusion matrix và curves

Tất cả metric dưới đây được tính trên `split='val'`. `rider_without_helmet` là metric ưu tiên số 1; confusion matrix được đọc từ object kết quả của Ultralytics.


In [ ]:
def safe_f1(precision, recall):
    return 2.0 * precision * recall / (precision + recall) if precision + recall else 0.0

def extract_metrics(result, label):
    summary = {
        'Precision': float(result.box.mp),
        'Recall': float(result.box.mr),
        'mAP50': float(result.box.map50),
        'mAP50-95': float(result.box.map),
    }
    summary['F1'] = safe_f1(summary['Precision'], summary['Recall'])
    summary['by_class'] = {}
    for class_id, class_name in enumerate(CLASS_NAMES):
        precision, recall, ap50, ap5095 = result.box.class_result(class_id)
        values = {
            'Precision': float(precision),
            'Recall': float(recall),
            'F1': safe_f1(float(precision), float(recall)),
            'AP50': float(ap50),
            'AP50-95': float(ap5095),
        }
        summary['by_class'][class_name] = values
    print(f'--- {label} overall ---')
    for name in ('Precision', 'Recall', 'F1', 'mAP50', 'mAP50-95'):
        print(f'{name:10s}: {summary[name]:.4f}')
    print('--- Per-class metrics ---')
    for class_name, values in summary['by_class'].items():
        print(class_name, ' | ', ' | '.join(f'{key}={values[key]:.4f}' for key in ('Precision', 'Recall', 'F1', 'AP50', 'AP50-95')))
    print('--- rider_without_helmet ---')
    print(summary['by_class']['rider_without_helmet'])
    return summary

validation_metrics = best_model.val(data=str(DATA_YAML_PATH), split=VALIDATION_SPLIT, imgsz=640, device=0, plots=True)
validation_summary = extract_metrics(validation_metrics, 'E6 validation')
no_helmet_metrics = validation_summary['by_class']['rider_without_helmet']
test_metrics = None
test_summary = None

def confusion_array(result):
    matrix = getattr(getattr(result, 'confusion_matrix', None), 'matrix', None)
    matrix = None if matrix is None else np.asarray(matrix)
    expected_shape = (len(CLASS_NAMES) + 1, len(CLASS_NAMES) + 1)
    return matrix if matrix is not None and matrix.shape == expected_shape else None

def confusion_error_counts(result):
    matrix = confusion_array(result)
    if matrix is None:
        return None

    # Ultralytics orientation used here: matrix[predicted_class, actual_class].
    labels = CLASS_NAMES + ['background']
    index = {label: position for position, label in enumerate(labels)}
    nh = index['rider_without_helmet']
    return {
        'rider_without_helmet -> rider_without_helmet': float(matrix[nh, nh]),
        'rider_without_helmet -> motorcycle': float(matrix[index['motorcycle'], nh]),
        'rider_without_helmet -> rider_with_helmet': float(matrix[index['rider_with_helmet'], nh]),
        'rider_without_helmet -> background': float(matrix[index['background'], nh]),
        'background -> rider_without_helmet': float(matrix[nh, index['background']]),
    }


def normalized_nohelmet_diagonal_from_result(result):
    matrix = confusion_array(result)
    if matrix is None:
        return None

    matrix = matrix.astype(float)
    nh = CLASS_NAMES.index('rider_without_helmet')
    total_actual_nohelmet = matrix[:, nh].sum()
    if total_actual_nohelmet == 0:
        return None
    return float(matrix[nh, nh] / total_actual_nohelmet)
confusion_errors = confusion_error_counts(validation_metrics)
no_helmet_diagonal_normalized = normalized_nohelmet_diagonal_from_result(validation_metrics)
print('NoHelmet normalized confusion diagonal:', no_helmet_diagonal_normalized)
if confusion_errors is not None:
    display(pd.DataFrame({'Error': list(confusion_errors), 'Count': list(confusion_errors.values())}))
    matrix = confusion_array(validation_metrics).astype(float)
    nh = CLASS_NAMES.index('rider_without_helmet')
    print('NoHelmet actual total:', matrix[:, nh].sum())
    print('NoHelmet correct:', matrix[nh, nh])
    assert no_helmet_diagonal_normalized is not None
    assert 0.0 <= no_helmet_diagonal_normalized <= 1.0
else:
    print('NoHelmet normalized confusion diagonal: unavailable from result object')
plot_specs = [
    ('results.png', 'Training results'),
    ('confusion_matrix.png', 'Confusion matrix'),
    ('confusion_matrix_normalized.png', 'Normalized confusion matrix'),
    ('BoxPR_curve.png', 'Box PR curve'),
    ('BoxF1_curve.png', 'Box F1 curve'),
    ('BoxP_curve.png', 'Box Precision curve'),
    ('BoxR_curve.png', 'Box Recall curve'),
]
for filename, title in plot_specs:
    paths = list(EXPERIMENT_DIR.rglob(filename)) if EXPERIMENT_DIR.exists() else []
    if paths:
        path = max(paths, key=lambda item: item.stat().st_mtime)
        plt.figure(figsize=(16, 10) if filename == 'results.png' else (10, 8))
        plt.imshow(Image.open(path))
        plt.axis('off')
        plt.title(title)
        plt.show()


## 9. Complexity trên cùng validation images

So sánh E0/E4/E5/E6 với `imgsz=640`, cùng GPU và tối đa 50 ảnh validation: parameters, GFLOPs, model size, latency và FPS.


In [ ]:
def extract_gflops(info):
    if isinstance(info, dict):
        for key in ('GFLOPs', 'gflops', 'flops'):
            if key in info:
                try:
                    return float(info[key])
                except (TypeError, ValueError):
                    pass
    if isinstance(info, (tuple, list)):
        for value in reversed(info):
            try:
                return float(value)
            except (TypeError, ValueError):
                pass
    return None

def benchmark_inference(yolo_model, image_paths):
    for image_path in image_paths[:2]:
        yolo_model.predict(source=image_path, imgsz=640, device=0, save=False, verbose=False)
    elapsed = []
    for image_path in image_paths:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_time = time.perf_counter()
        yolo_model.predict(source=image_path, imgsz=640, device=0, save=False, verbose=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed.append(time.perf_counter() - start_time)
    milliseconds = float(np.mean(elapsed) * 1000.0)
    return milliseconds, 1000.0 / milliseconds if milliseconds else 0.0

def build_custom_scconv_from_checkpoint(path):
    return load_arch_checkpoint(
        path, build_custom_scconv_model,
        lambda module: count_c3k2_scconv(module) == 2,
        'E5',
    )[0]

def build_custom_dysample_from_checkpoint(path):
    return load_arch_checkpoint(
        path, build_custom_dysample_model,
        lambda module: count_dysample(module) == 2,
        'E4',
    )[0]

valid_img_dir = Path('/content/DetectedHelmets/DataSet2/DataSet2/valid/images')
valid_images = sorted(
    str(path) for pattern in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
    for path in valid_img_dir.glob(pattern)
)
assert valid_images, valid_img_dir
speed_images = valid_images[:min(50, len(valid_images))]

assert BASELINE_BEST_MODEL_PATH.exists(), BASELINE_BEST_MODEL_PATH
baseline_model = YOLO(str(BASELINE_BEST_MODEL_PATH))
e4_model = build_custom_dysample_from_checkpoint(E4_BEST_MODEL_PATH)
e5_model = build_custom_scconv_from_checkpoint(E5_BEST_MODEL_PATH)
assert count_dysample(e4_model.model) == 2
assert count_c3k2_scconv(e5_model.model) == 2

complexity_rows = []
complexity_models = [
    ('E0', baseline_model, BASELINE_BEST_MODEL_PATH),
    ('E4', e4_model, E4_BEST_MODEL_PATH),
    ('E5', e5_model, E5_BEST_MODEL_PATH),
    ('E6', best_model, BEST_MODEL_PATH),
]
for label, candidate, checkpoint_path in complexity_models:
    latency, fps = benchmark_inference(candidate, speed_images)
    complexity_rows.append({
        'Model': label,
        'Parameters': sum(parameter.numel() for parameter in candidate.model.parameters()),
        'GFLOPs': extract_gflops(candidate.info(verbose=True, imgsz=640)),
        'Model size (MB)': checkpoint_path.stat().st_size / (1024 ** 2),
        'Inference ms/image': latency,
        'FPS': fps,
    })
complexity = pd.DataFrame(complexity_rows).set_index('Model')
display(complexity)


## 10. So sánh E0 / E4 / E5 / E6 trên cùng validation set

Bảng gồm metric overall, AP50 từng class, metric ưu tiên no-helmet và diagonal normalized của no-helmet. Quyết định KEEP/REJECT chỉ dùng số đo thực tế, không hard-code kết quả.


In [ ]:
e0_metrics = baseline_model.val(data=str(DATA_YAML_PATH), split=VALIDATION_SPLIT, imgsz=640, device=0, plots=False)
e4_metrics = e4_model.val(data=str(DATA_YAML_PATH), split=VALIDATION_SPLIT, imgsz=640, device=0, plots=False)
e5_metrics = e5_model.val(data=str(DATA_YAML_PATH), split=VALIDATION_SPLIT, imgsz=640, device=0, plots=False)
e6_metrics = validation_metrics
e0_summary = extract_metrics(e0_metrics, 'E0 validation')
e4_summary = extract_metrics(e4_metrics, 'E4 validation')
e5_summary = extract_metrics(e5_metrics, 'E5 validation')

def summary_values(summary, diagonal=None):
    return {
        'Precision': summary['Precision'],
        'Recall': summary['Recall'],
        'F1': summary['F1'],
        'mAP50': summary['mAP50'],
        'mAP50-95': summary['mAP50-95'],
        'Motorcycle AP50': summary['by_class']['motorcycle']['AP50'],
        'WithHelmet AP50': summary['by_class']['rider_with_helmet']['AP50'],
        'NoHelmet AP50': summary['by_class']['rider_without_helmet']['AP50'],
        'NoHelmet Precision': summary['by_class']['rider_without_helmet']['Precision'],
        'NoHelmet Recall': summary['by_class']['rider_without_helmet']['Recall'],
        'NoHelmet F1': summary['by_class']['rider_without_helmet']['F1'],
        'NoHelmet AP50-95': summary['by_class']['rider_without_helmet']['AP50-95'],
        'NoHelmet normalized confusion diagonal': diagonal,
    }

e0_errors = confusion_error_counts(e0_metrics)
e4_errors = confusion_error_counts(e4_metrics)
e5_errors = confusion_error_counts(e5_metrics)

e0_diagonal = normalized_nohelmet_diagonal_from_result(e0_metrics)
e4_diagonal = normalized_nohelmet_diagonal_from_result(e4_metrics)
e5_diagonal = normalized_nohelmet_diagonal_from_result(e5_metrics)
e6_diagonal = normalized_nohelmet_diagonal_from_result(validation_metrics)

for model_label, result, diagonal in (
    ('E0', e0_metrics, e0_diagonal),
    ('E4', e4_metrics, e4_diagonal),
    ('E5', e5_metrics, e5_diagonal),
    ('E6', validation_metrics, e6_diagonal),
):
    matrix = confusion_array(result)
    if matrix is not None:
        nh = CLASS_NAMES.index('rider_without_helmet')
        print(model_label, 'NoHelmet actual total:', matrix[:, nh].sum())
        print(model_label, 'NoHelmet correct:', matrix[nh, nh])
        if diagonal is None:
            print(model_label, 'NoHelmet diagonal: undefined (no actual rider_without_helmet ground truth in matrix)')
        else:
            assert 0.0 <= diagonal <= 1.0
comparison_values = {
    'E0': summary_values(e0_summary, e0_diagonal),
    'E4': summary_values(e4_summary, e4_diagonal),
    'E5': summary_values(e5_summary, e5_diagonal),
    'E6': summary_values(validation_summary, e6_diagonal),
}
comparison = pd.DataFrame(comparison_values).loc[list(next(iter(comparison_values.values())).keys())]
display(comparison)

error_comparison = None
if all(errors is not None for errors in (e0_errors, e4_errors, e5_errors, confusion_errors)):
    error_keys = list(confusion_errors.keys())
    error_comparison = pd.DataFrame({
        'Error': error_keys,
        'E0': [e0_errors[key] for key in error_keys],
        'E4': [e4_errors[key] for key in error_keys],
        'E5': [e5_errors[key] for key in error_keys],
        'E6': [confusion_errors[key] for key in error_keys],
    }).set_index('Error')
    display(error_comparison)

def value(model_name, metric):
    raw = comparison.loc[metric, model_name]
    return None if pd.isna(raw) else float(raw)

priority_improvements = {
    'mAP50-95_vs_E0': value('E6', 'mAP50-95') >= value('E0', 'mAP50-95'),
    'NoHelmet_AP50_vs_E5': value('E6', 'NoHelmet AP50') > value('E5', 'NoHelmet AP50'),
    'NoHelmet_Recall_vs_E5': value('E6', 'NoHelmet Recall') > value('E5', 'NoHelmet Recall'),
    'NoHelmet_F1_vs_E5': value('E6', 'NoHelmet F1') > value('E5', 'NoHelmet F1'),
}
mAP50_better_than_e0 = value('E6', 'mAP50') > value('E0', 'mAP50')
diagonal_better_than_e5 = (
    value('E6', 'NoHelmet normalized confusion diagonal') is not None
    and value('E5', 'NoHelmet normalized confusion diagonal') is not None
    and value('E6', 'NoHelmet normalized confusion diagonal') > value('E5', 'NoHelmet normalized confusion diagonal')
)
e6_keep = mAP50_better_than_e0 and (any(priority_improvements.values()) or diagonal_better_than_e5)
e6_decision = 'KEEP E6' if e6_keep else 'REJECT E6'
print('Priority improvements:', priority_improvements)
print('mAP50 > E0:', mAP50_better_than_e0)
print('NoHelmet diagonal > E5:', diagonal_better_than_e5)
print('E6 decision:', e6_decision)


## 11. Final audit

Audit cuối cùng không giả định E6 đạt mục tiêu; nó chỉ xác nhận tính đúng đắn của experiment và in quyết định dựa trên số đo.


In [ ]:
assert ultralytics.__version__ == '8.4.155'
assert DATA_YAML_PATH == Path('/content/DetectedHelmets/DataSet2/DataSet2/data.yaml')
assert CLASS_NAMES == EXPECTED_CLASS_NAMES
assert TRAIN_CONFIG == {
    'epochs': 50, 'imgsz': 640, 'batch': 16, 'device': 0, 'workers': 4,
    'optimizer': 'auto', 'seed': 0, 'deterministic': True, 'patience': 15,
    'pretrained': True,
}
assert VALIDATION_SPLIT == 'val'
assert DYSAMPLE_EXPECTED_COUNT == 2
assert C3K2_SCCONV_EXPECTED_COUNT == 2
assert count_dysample(custom_pytorch_model) == 2
assert count_c3k2_scconv(custom_pytorch_model) == 2
assert count_dysample(trainer.model) == 2
assert count_c3k2_scconv(trainer.model) == 2
assert dysample_count_best == 2
assert c3k2_scconv_count_best == 2
assert dysample_grad_ok and scconv_grad_ok
assert test_metrics is None and test_summary is None
assert EXPERIMENT_DIR.exists() and BEST_MODEL_PATH.exists()
assert EXPERIMENT_NAME == 'helmet_yolo26s_scconv_dysample_dataset2'
assert EXPERIMENT_VERSION_FILE.exists()
assert 'ultralytics==8.4.155' in EXPERIMENT_VERSION_FILE.read_text(encoding='utf-8')

forbidden_modules = {'P2', 'iAFF', 'MultiSEAM', 'SEAM', 'CBAM', 'ECA', 'AFPN', 'ADown'}
custom_module_names = {item.__class__.__name__ for item in best_model.model.modules()}
assert not (custom_module_names & forbidden_modules), custom_module_names & forbidden_modules

print('Final audit passed.')
print('Ultralytics:', ultralytics.__version__)
print('DySample before/train/best:', dysample_before_train, dysample_inside_trainer, dysample_count_best)
print('C3k2_SCConv before/train/best:', c3k2_scconv_before_train, c3k2_scconv_inside_trainer, c3k2_scconv_count_best)
print('SCConv gradient:', scconv_grad_ok)
print('NoHelmet metrics:', no_helmet_metrics)
print('Validation only; test split not used.')
print('E6 checkpoint:', BEST_MODEL_PATH)
print('Decision from measured metrics:', e6_decision)
